# Practical 1 - Biological Data Retrieval and proteome stats for Global Health Proteomes

During today's practical you'll need to get an overview of your assigned organism as part of the Global Health Proteomes initiative. The practical is self guided and it won't be marked towards your final exam score.

Create code or text cells as you go along and at the end, save it with "File --> Download --> as ipynb" and send it to my email address (nicola.bordin.1@unipd.it)

**Describe briefly your assigned organism.**







Access the proteins for your assigned TaxonID at https://www.uniprot.org/taxonomy?query=*

For example, for TaxonID 36087 the results will be at

https://www.uniprot.org/taxonomy/36087

**How many entries in UniProt are associated to your TaxonID?**

**Let's access the proteome information by clicking "View Proteomes**".

**How many proteins are included in the reference proteome, if available?**

**Why do you think we have less proteins in the reference proteome than the grand total we find for this organism in UniProt?**


**How many proteins are manually annotated?**

**How many proteins are automatically annotated?**

On the proteome overview page, please right click and copy the link on "Download one protein sequence per gene (FASTA)".

This will give you a link that looks like this

https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/reference_proteomes/Eukaryota/UP000030665/UP000030665_36087.fasta.gz

Replace the link below with your link

Then extract the zipped FASTA file using gunzip.

In [ ]:
!wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/reference_proteomes/Eukaryota/UP000030665/UP000030665_36087.fasta.gz
!gunzip UP000030665_36087.fasta.gz

Can you see your file locally? Where are we on the OS?

**HINT: On Google Colab any non-Python command (such as cat, less, grep) needs to be prepended with an exclamation mark, as such !grep !cat !less**

**How many lines we have in our FASTA file?**

**How many proteins do we have in our FASTA file?**

In [ ]:
!grep ">" UP000030665_36087.fasta

**Try to exclude the FASTA headers and count how many aminoacid this proteome encodes. Two flags can help with this: `-v` in `grep` and `-m` for `wc`.
Can you combine both commands into a very simple pipeline?**

In [ ]:
!grep -v ">" UP000030665_36087.fasta | wc -m

Let's introduce a new linux command: `tr`

`tr` lets us do **t**ext **r**eplacement

So we can also instruct `tr` to replace all newline characters with nothing so we concatenate all aminoacids in our sequences.

In [ ]:
!grep -v ">" UP000030665_36087.fasta | tr -d '\n' | wc -m

Why do we get two different results?

Let's move to Python



In [ ]:
filename = "UP000030665_36087.fasta"
headers = []
sequences = []

current_header = ""
current_sequence = ""

with open(filename, "rt") as fh:
    for line in fh:
        line = line.strip()

        if line.startswith(">"):
            if current_sequence != "":
                headers.append(current_header)
                sequences.append(current_sequence)
            current_header = line
            current_sequence = ""
        else:
            current_sequence += line

# add last entry
if current_sequence != "":
    headers.append(current_header)
    sequences.append(current_sequence)

print("Loaded", len(sequences), "proteins")

Let's calculate the aminoacid composition of our organism

In [ ]:
aa_counts = {}

for seq in sequences:
    for aa in seq:
        if aa not in aa_counts:
            aa_counts[aa] = 0
        aa_counts[aa] += 1

print(aa_counts)

Let's normalise the frequencies

In [ ]:
total = sum(aa_counts.values())

aa_freq = {}

for aa in aa_counts:
    aa_freq[aa] = aa_counts[aa] / total

for aa in sorted(aa_freq):
    print(aa, round(aa_freq[aa], 4))

Let's plot the frequencies

In [ ]:
import matplotlib.pyplot as plt

aas = sorted(aa_freq.keys())
freqs = [aa_freq[a] for a in aas]

plt.bar(aas, freqs)
plt.xlabel("Amino acid")
plt.ylabel("Frequency")
plt.title("Amino acid composition")
plt.show()

What does the code below do?

In [ ]:
lengths = [len(seq) for seq in sequences]

min_len = min(lengths)
max_len = max(lengths)

print("Shortest protein:", min_len)
print("Longest protein:", max_len)

Let's print the shorter proteins in our proteome.

In [ ]:
pairs = list(zip(headers, sequences))
pairs.sort(key=lambda x: len(x[1]))

print("5 shortest proteins:\n")
for h, s in pairs[:5]:
    print(len(s), h)

How many proteins are "Uncharacterized"? What's the proportion over the total number of sequences in our organism?

**What's the mean length of our proteins?**




In [ ]:
mean_length = sum(len(s) for s in sequences) / len(sequences)
print("Average protein length:", round(mean_length, 2))

Do most proteins have similar lengths? Or is there a long tail?

Are very large proteins common in this species?

Which aminoacids dominate the proteome?

Why might some sequences contain X?

# Let's head back to UniProt and download the annotations

You should be able to customise the look of your table. Please include the following columns

Entry

Reviewed

Entry Name

Gene Names

Organism

Length

Mass

Taxonomic lineage

Protein existence

Function [CC]

EC number

Keywords

Gene Ontology (biological process)

Gene Ontology (cellular component)

Gene Ontology (molecular function)

Subcellular location [CC]

Transmembrane

Once done, click Download, format TSV, compressed NO. Instead of left-clicking download, do right-click and "copy link". We will use that to dowload the table directly into our Google Colab.

My link looks like

https://rest.uniprot.org/uniprotkb/stream?compressed=true&download=true&fields=accession%2Creviewed%2Cid%2Cgene_names%2Corganism_name%2Clength%2Cmass%2Clineage%2Cprotein_existence%2Ccc_function%2Cec%2Ckeyword%2Cgo_p%2Cgo_c%2Cgo_f%2Ccc_subcellular_location%2Cft_transmem&format=tsv&query=%28proteome%3AUP000030665%29

Let's feed this to `wget`

In [ ]:
your_link_here = "https://rest.uniprot.org/uniprotkb/stream?download=true&fields=accession%2Creviewed%2Cid%2Cgene_names%2Corganism_name%2Clength%2Cmass%2Clineage%2Cprotein_existence%2Ccc_function%2Cec%2Ckeyword%2Cgo_p%2Cgo_c%2Cgo_f%2Ccc_subcellular_location%2Cft_transmem&format=tsv&query=%28proteome%3AUP000030665%29"

In [ ]:
!wget -O proteome.tsv "$your_link_here"

Let's inspect the tsv we just downloaded

In [ ]:
with open("proteome.tsv") as fh:
    header = fh.readline().strip().split("\t")

for i, col in enumerate(header):
    print(i, col)

How many reviewed proteins (e.g. Manually curated) are in our dataset?

In [ ]:
with open("proteome.tsv") as fh:
    header = fh.readline().strip().split("\t")

col = {name: i for i, name in enumerate(header)}

print(col)

reviewed = 0
total = 0

with open("proteome.tsv") as fh:
    fh.readline()  # skip header

    for line in fh:
        fields = line.strip().split("\t")
        total += 1

        if fields[col["Reviewed"]] == "reviewed":
            reviewed += 1

print("Reviewed:", reviewed)
print("Total:", total)
print("Fraction reviewed:", round(reviewed / total, 3))

In [ ]:
with_function = 0

with open("proteome.tsv") as fh:
    fh.readline()

    for line in fh:
        fields = line.strip().split("\t")

        if fields[col["Function [CC]"]] != "":
            with_function += 1

print("Proteins with annotated function:", with_function)

In [ ]:
enzymes = 0

with open("proteome.tsv") as fh:
    fh.readline()

    for line in fh:
        fields = line.strip().split("\t")

        if fields[col["EC number"]] != "":
            enzymes += 1

print("Proteins with EC numbers:", enzymes)
print("Fraction:", round(enzymes / total, 3))

In [ ]:
annotated = 0

with open("proteome.tsv") as fh:
    fh.readline()

    for line in fh:
        fields = line.rstrip("\n").split("\t")

        go_bp = fields[col["Gene Ontology (biological process)"]]
        go_mf = fields[col["Gene Ontology (molecular function)"]]
        go_cc = fields[col["Gene Ontology (cellular component)"]]

        if go_bp != "" or go_mf != "" or go_cc != "":
            annotated += 1

print("Proteins with GO annotation:", annotated)
print("Fraction:", round(annotated / total, 3))

Using both your FASTA file and the UniProt TSV file, investigate whether long proteins are better annotated than short proteins.

Specifically:

1. Define:
    * Short proteins: length < 300 amino acids
    * Long proteins: length > 800 amino acids
2. For each group, calculate:
    * the number of proteins
    * the fraction that have GO annotation (any of BP, MF, or CC)
3. Compare the two groups and answer:
    * Are long proteins more likely to be annotated?
    * Why might this be the case?

In [ ]:
length_dict = {}

for h, s in zip(headers, sequences):
    protein_id = h.split("|")[1]
    length_dict[protein_id] = len(s)

In [ ]:
short_total = 0
short_annot = 0

long_total = 0
long_annot = 0

with open("proteome.tsv") as fh:
    header = fh.readline().rstrip("\n").split("\t")

    col = {name: i for i, name in enumerate(header)}

    for line in fh:
        fields = line.rstrip("\n").split("\t")

        entry = fields[col["Entry"]]

        if entry not in length_dict:
            continue

        length = length_dict[entry]

        go_bp = fields[col["Gene Ontology (biological process)"]]
        go_mf = fields[col["Gene Ontology (molecular function)"]]
        go_cc = fields[col["Gene Ontology (cellular component)"]]

        has_go = (go_bp != "" or go_mf != "" or go_cc != "")

        if length < 300:
            short_total += 1
            if has_go:
                short_annot += 1

        elif length > 800:
            long_total += 1
            if has_go:
                long_annot += 1

print("Short proteins:", short_total)
print("Annotated (short):", short_annot)
print("Fraction:", round(short_annot / short_total, 3))

print()

print("Long proteins:", long_total)
print("Annotated (long):", long_annot)
print("Fraction:", round(long_annot / long_total, 3))